In [1]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

from src.data.make_db import build_db
from src.data.sql_runner import run_query, same_result

In [2]:
conn = build_db()

for table in ["customers", "products", "orders"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(table, count)

# Different spelling, same meaning: expect True
print(same_result(conn, "SELECT COUNT(*) FROM orders", "SELECT COUNT(id) FROM orders"))

# Different meaning: expect False
print(same_result(conn, "SELECT COUNT(*) FROM orders", "SELECT COUNT(*) FROM customers"))

# Broken SQL and dangerous SQL: expect None for both
print(run_query(conn, "SELEC nonsense"))
print(run_query(conn, "DROP TABLE orders"))

# The dangerous query must have done nothing: expect 400
print(conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0])

customers 40
products 25
orders 400
True
False
None
None
400


In [1]:
import json

def load(name):
    return [json.loads(line) for line in open(f"../data/raw/{name}.jsonl", encoding="utf-8")]

train, val, test = load("train"), load("val"), load("test")
print(len(train), len(val), len(test))

# Leakage check: no SQL may appear in two splits
train_sql = {r["sql"] for r in train}
assert not train_sql & {r["sql"] for r in val}
assert not train_sql & {r["sql"] for r in test}
print("no leakage")

for row in train[:5]:
    print(row["question"], "->", row["sql"])

1054 132 132
no leakage
average price in the clothing category -> SELECT AVG(price) FROM products WHERE category = 'clothing'
average price in the electronics category -> SELECT AVG(price) FROM products WHERE category = 'electronics'
average price in the kitchen category -> SELECT AVG(price) FROM products WHERE category = 'kitchen'
average price in the stationery category -> SELECT AVG(price) FROM products WHERE category = 'stationery'
average price in the toys category -> SELECT AVG(price) FROM products WHERE category = 'toys'


In [4]:
import re

def skeleton(sql):
    """Mask the values so every SQL from the same template looks identical."""
    sql = re.sub(r"'[^']*'", "?", sql)           # quoted strings -> ?
    return re.sub(r"\b\d+(\.\d+)?\b", "?", sql)  # numbers -> ?

all_skeletons = {skeleton(r["sql"]) for r in train + val + test}
print("templates found:", len(all_skeletons))     # expect 12

for name, rows in [("train", train), ("val", val), ("test", test)]:
    present = {skeleton(r["sql"]) for r in rows}
    missing = all_skeletons - present
    print(f"{name}: missing {len(missing)} templates")
    for m in missing:
        print("   ", m[:80])

templates found: 12
train: missing 0 templates
val: missing 3 templates
    SELECT name FROM products WHERE category = ?
    SELECT AVG(price) FROM products WHERE category = ?
    SELECT customers.city, COUNT(*) FROM orders JOIN customers ON orders.customer_id
test: missing 3 templates
    SELECT name FROM products WHERE category = ?
    SELECT AVG(price) FROM products WHERE category = ?
    SELECT customers.city, COUNT(*) FROM orders JOIN customers ON orders.customer_id


In [5]:
import json, re

def load(name):
    return [json.loads(line) for line in open(f"../data/raw/{name}.jsonl", encoding="utf-8")]

def skeleton(sql):
    sql = re.sub(r"'[^']*'", "?", sql)
    return re.sub(r"\b\d+(\.\d+)?\b", "?", sql)

train, val, test = load("train"), load("val"), load("test")
print(len(train), len(val), len(test))

# Leakage check
train_sql = {r["sql"] for r in train}
assert not train_sql & {r["sql"] for r in val}
assert not train_sql & {r["sql"] for r in test}
print("no leakage")

# Coverage check
all_skeletons = {skeleton(r["sql"]) for r in train + val + test}
print("templates found:", len(all_skeletons))
for name, rows in [("train", train), ("val", val), ("test", test)]:
    missing = all_skeletons - {skeleton(r["sql"]) for r in rows}
    print(f"{name}: missing {len(missing)} templates")

1049 135 134
no leakage
templates found: 12
train: missing 0 templates
val: missing 0 templates
test: missing 0 templates
